# 01 · Dati: download, merge, pulizia — versione multi-fonte

Il notebook costruisce quattro cose:
1. una tabella di **proteine** con la sequenza;
2. una tabella **lunga di osservazioni**: per ogni proteina, ogni fonte e ogni compartimento, la fonte dice
   1 (presente), 0 (assente) oppure niente (non misurato);
3. i **target ausiliari** (meccanismi di sorting e PTM) e il **contesto genico**: sono le caratteristiche che il vecchio
   notebook FFFFF ricavava da UniProt, HPA, MitoCarta e SynaptopathyDB, ora calcolate direttamente qui;
4. le **varianti** ClinVar e, se disponibili, quelle con localizzazione misurata sperimentalmente.

**Perché una tabella di osservazioni invece di un'etichetta unica.** Le fonti hanno affidabilità diverse e note:
DeepLoc (sperimentale, curata) è l'ancora; UniProt è incompleto; HPA Approved è meno solido di Enhanced; le mappe
neuronali di topo sono un'altra specie. Invece di fonderle a mano con soglie arbitrarie, il notebook 02 impara per
ogni fonte e classe **sensibilità** (quante localizzazioni vere riporta) e **specificità** (quanti falsi positivi).
Così si usano anche le etichette deboli senza farsi ingannare.

**Regola delle assenze.** Per ogni fonte, una proteina che la fonte riporta e un compartimento che la fonte sa
misurare ma non assegna a quella proteina = osservazione 0. Tutto il resto = non osservato. Esempio: HPA non misura
l'extracellulare, quindi per HPA quella classe resta sempre vuota.

**Fonti automatiche:** DeepLoc 2.0 (training + test HPA indipendente), UniProt umano, HPA subcellular,
HPA RNA tessuti e single-cell, OpenCell, MitoCarta 3.0, SynaptopathyDB, ClinVar.

**Cosa è rimasto di FFFFF e cosa no.** Restano: query UniProt cerebrale, filtro di espressione cerebrale HPA,
MitoCarta, SynaptopathyDB, ClinVar, le feature di PTM / NLS / multimero / aggregati / patologia cerebrale / ruolo
di trafficking / struttura PDB (ora dal campo UniProt, senza interrogare RCSB). Sono state tolte: le ~9000 colonne
GO (contengono la risposta), COMPARTMENTS (integra UniProt e GO, quindi circolare), Allen (sostituito dai dati HPA
di tessuto e single-cell), PDB via RCSB, le celle "PIR" e "SynGO" di FFFFF (erano query UniProt ripetute; il vero SynGO ora entra come fonte), synaptosome ed ExoCarta
(le pagine scaricate non contenevano dati).

**Fonti da scaricare a mano** (tutte facoltative; istruzioni passo per passo nel README):
- `data/esterni/syngo_ontologies.xlsx` — SynGO (syngoportal.org, "bulk download"): il notebook lo converte da solo
  in presinapsi/postsinapsi;
- `data/esterni/fonti/<nome>.csv` — altre mappe di localizzazione già convertite in due colonne `gene`, `classe`
  (es. neuroni di topo, Itzhak 2017; assone/somatodendritico, Hobson 2022). Il nome del file = nome della fonte;
- `data/esterni/varianti/<nome>.csv` — varianti con localizzazione misurata (es. Lacoste 2024), colonne `gene`,
  `variante` (es. `R273H`), `mislocalizzata` (0/1). Servono solo per la valutazione finale.
Se mancano, il notebook funziona lo stesso.

Output: `data/proteine.parquet`, `data/osservazioni.parquet`, `data/clinvar_missense.parquet`,
`data/varianti_sperimentali.parquet`

In [3]:
import io
import gzip
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

CACHE, DATA, EXT = Path('cache'), Path('data'), Path('data/esterni')
for d in [CACHE, DATA, EXT / 'fonti', EXT / 'varianti']:
    d.mkdir(parents=True, exist_ok=True)

CLASSES = ['Cytoplasm', 'Nucleus', 'Extracellular', 'Cell membrane', 'Mitochondrion',
           'Endoplasmic reticulum', 'Lysosome/Vacuole', 'Golgi apparatus', 'Peroxisome']
NEURO = ['Assone', 'Dendrite', 'Presinapsi', 'Postsinapsi']
BRAIN_NTPM, NEURON_NCPM = 10, 5

session = requests.Session()
session.headers['User-Agent'] = 'brain-misloc/3.0 (academic research; UNISA)'
session.mount('https://', HTTPAdapter(max_retries=Retry(total=6, backoff_factor=2,
                                                        status_forcelist=[429, 500, 502, 503, 504])))

In [4]:
UNIPROT_FIELDS = ('accession,gene_names,protein_name,sequence,cc_subcellular_location,xref_geneid,'
                  'ft_signal,ft_transit,ft_transmem,ft_lipid,keyword,cc_ptm,cc_function,cc_disease,cc_subunit,'
                  'ft_region,ft_motif,xref_pdb')
BRAIN_QUERY = ('reviewed:true AND organism_id:9606 AND (cc_tissue_specificity:brain OR cc_tissue_specificity:neuron '
               'OR cc_tissue_specificity:neuro* OR cc_tissue_specificity:cerebr* OR cc_tissue_specificity:hippocampus '
               'OR cc_tissue_specificity:cortex OR cc_tissue_specificity:astrocyte OR cc_tissue_specificity:microglia '
               'OR cc_tissue_specificity:glia OR cc_tissue_specificity:"nervous system" OR keyword:KW-0165 '
               'OR keyword:KW-0523 OR keyword:KW-0524 OR go:0007268 OR go:0030182 OR go:0007411 OR go:0045202 '
               'OR go:0043005 OR go:0098794 OR go:0014069)')
STREAM = 'https://rest.uniprot.org/uniprotkb/stream'
DTU = 'https://services.healthtech.dtu.dk/services/DeepLoc-2.0/data/'
HPA = 'https://www.proteinatlas.org/download/tsv/'

sources = pd.DataFrame([
    ('deeploc_train.csv', DTU + 'Swissprot_Train_Validation_dataset.csv', None),
    ('deeploc_hpa_test.csv', DTU + 'hpa_testset.csv', None),
    ('uniprot_human_v4.tsv', STREAM, {'query': 'reviewed:true AND organism_id:9606 AND fragment:false',
                                      'fields': UNIPROT_FIELDS, 'format': 'tsv'}),
    ('uniprot_brain_ids.tsv', STREAM, {'query': BRAIN_QUERY, 'fields': 'accession', 'format': 'tsv'}),
    ('hpa_subcellular.zip', HPA + 'subcellular_location.tsv.zip', None),
    ('hpa_rna_tissue.zip', HPA + 'rna_tissue_consensus.tsv.zip', None),
    ('hpa_rna_single_cell.zip', HPA + 'rna_single_cell_type.tsv.zip', None),
    ('synaptopathydb_genes.csv', 'https://www.synaptopathydb.org/api/genes', None),
    ('opencell_localization.csv', 'https://opencell.sf.czbiohub.org/data/datasets/opencell-localization-annotations.csv', None),
    # MitoCarta: il sito Broad blocca i download automatici (403 Akamai).
    # Scarica manualmente Human.MitoCarta3.0.xls dalla pagina
    # https://www.broadinstitute.org/mitocarta/mitocarta30-inventory-mammalian-mitochondrial-proteins-and-pathways
    # e salvalo come  cache/mitocarta3.xlsx  — oppure lascia perdere: il fallback usa UniProt KW-0496.
    ('clinvar_variant_summary.txt.gz', 'https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz', None),
], columns=['file', 'url', 'params'])

# scarico solo i file mancanti; il .part evita file troncati in cache se il download si interrompe
missing = sources[[not (CACHE / f).exists() for f in sources['file']]]
for f, url, params in missing.itertuples(index=False):
    try:
        with session.get(url, params=params, stream=True, timeout=3600) as r, open(CACHE / f'{f}.part', 'wb') as out:
            r.raise_for_status()
            for chunk in r.iter_content(1 << 20):
                out.write(chunk)
        (CACHE / f'{f}.part').replace(CACHE / f)
        print('scaricato', f)
    except Exception as e:
        import warnings; warnings.warn(f'download saltato ({f}): {e}')
        try: (CACHE / f'{f}.part').unlink(missing_ok=True)
        except Exception: pass

## Proteine: DeepLoc (training con partizioni di omologia), test HPA, UniProt umano

In [5]:
dl = pd.read_csv(CACHE / 'deeploc_train.csv', index_col=0).rename(
    columns={'ACC': 'uniprot_id', 'Sequence': 'sequence', 'Partition': 'partition'})
dl['source'] = 'deeploc'
hpa_test = pd.read_csv(CACHE / 'deeploc_hpa_test.csv').rename(columns={'sid': 'id_test', 'fasta': 'sequence'})
hpa_test['source'], hpa_test['partition'] = 'hpa_test', -1

up = pd.read_csv(CACHE / 'uniprot_human_v4.tsv', sep='\t', dtype=str).rename(columns={
    'Entry': 'uniprot_id', 'Gene Names': 'gene_names', 'Protein names': 'protein_name', 'Sequence': 'sequence',
    'Subcellular location [CC]': 'scl', 'GeneID': 'geneid', 'Signal peptide': 'sig', 'Transit peptide': 'transit',
    'Transmembrane': 'tm', 'Lipidation': 'lipid', 'Keywords': 'keywords', 'Post-translational modification': 'ptm',
    'Function [CC]': 'function', 'Involvement in disease': 'disease', 'Subunit structure': 'subunit', 'Region': 'region',
    'Motif': 'motif', 'PDB': 'pdb'})
up['gene'] = up['gene_names'].fillna('').str.split().str[0].str.upper()   # in UniProt i nomi sono separati da spazi
brain_ids = set(pd.read_csv(CACHE / 'uniprot_brain_ids.tsv', sep='\t')['Entry'])

# simbolo genico -> UniProt: simbolo ufficiale, poi alias solo se non ambigui (serve per HPA, MitoCarta, fonti esterne)
tokens = up[['uniprot_id', 'gene_names']].assign(sym=up['gene_names'].fillna('').str.upper().str.split()).explode('sym')
alias_n = tokens.groupby('sym')['uniprot_id'].transform('nunique')
sym2id = pd.concat([up.set_index('gene')['uniprot_id'], tokens[alias_n == 1].set_index('sym')['uniprot_id']])
sym2id = sym2id[~sym2id.index.duplicated() & (sym2id.index != '')]

# il test HPA usa ID Ensembl: lo collego a UniProt per sequenza identica
hpa_test = hpa_test.merge(up[['uniprot_id', 'sequence']].drop_duplicates('sequence'), on='sequence', how='left')

## Osservazioni da UniProt
Ogni frase del campo di localizzazione ha la forma `Termine {ECO:...}`. Il codice ECO dice l'evidenza:
sperimentale (269, 314, 7005, 353) o inferenza del curatore (305) → fonte `uniprot_forte`; per similarità o
predizione → fonte `uniprot_debole`. Le due fonti avranno sensibilità e specificità diverse.

In [6]:
scl = up.set_index('uniprot_id')['scl'].fillna('')
scl = scl.str.replace(r'SUBCELLULAR LOCATION: \[Isoform (?!1\])[^\]]*\]:.*?(?=SUBCELLULAR LOCATION:|$)', '', regex=True)
scl = scl.str.replace(r'Note=.*?(?=SUBCELLULAR LOCATION:|$)', '', regex=True)
scl = scl.str.replace(r'SUBCELLULAR LOCATION:|\[Isoform 1\]:', '.', regex=True)
terms = scl.str.extractall(r'([A-Za-z][^.;{}]*?)\s*\{([^}]*)\}').rename(columns={0: 'term', 1: 'eco'}).reset_index()
term = terms['term'].str.strip().str.lower()

# termine -> organello: le regole vanno dalla più generica alla più specifica, l'ultima che fa match vince
RULES = [
    ('Cytoplasm', r'cytoplasm|cytosol|cytoskeleton|centrosome|centriole|microtubule|perikaryon|actin|spindle|'
                  r'p-body|stress granule|myofibril|sarcomere|z line'),
    ('Nucleus', r'\bnucleus\b|nucleolus|nucleoplasm|chromosome|centromere|kinetochore|telomere|pml body|cajal|'
                r'nuclear (?:body|speckle|pore|envelope|membrane|matrix)'),
    ('Cell membrane', r'cell membrane|plasma membrane|cell surface|junction|sarcolemma|t-tubule|synaptic cell membrane'),
    ('Extracellular', r'^secreted|extracellular space|extracellular matrix|basement membrane'),
    ('Endoplasmic reticulum', r'endoplasmic reticulum|sarcoplasmic reticulum|microsome'),
    ('Golgi apparatus', r'golgi'),
    ('Mitochondrion', r'mitochondri'),
    ('Lysosome/Vacuole', r'lysosom|vacuol|melanosom|autophagosom'),
    ('Peroxisome', r'peroxisom'),
]
terms['classe'] = None
for cls, pattern in RULES:
    terms.loc[term.str.contains(pattern), 'classe'] = cls
# topologia di membrana, vescicole generiche, esosomi: non sono organelli ("Cytoplasmic vesicle" NON è citoplasma)
terms.loc[term.str.contains(r'single-pass|multi-pass|peripheral membrane|lipid-anchor|gpi-anchor|\bside$|^membrane$|'
                            r'vesicle|endosom|exosome|granule membrane'), 'classe'] = None

# termine -> compartimento neuronale (stesse frasi, regole proprie: qui le vescicole sinaptiche contano)
neuro_term = pd.Series(None, index=terms.index, dtype=object)
for cls, pattern in [('Assone', r'\baxon'), ('Dendrite', r'dendrit|spine'),
                     ('Presinapsi', r'presynap|synaptic vesicle|active zone|synaptic bouton'),
                     ('Postsinapsi', r'postsynap|postsynaptic density')]:
    neuro_term[term.str.contains(pattern)] = cls
terms = pd.concat([terms, terms.assign(classe=neuro_term)], ignore_index=True).dropna(subset=['classe'])
strong = terms['eco'].str.contains(r'ECO:0000269|ECO:0000314|ECO:0007005|ECO:0000353|ECO:0000305')
obs_uniprot = terms.assign(fonte=np.where(strong, 'uniprot_forte', 'uniprot_debole'))[['uniprot_id', 'fonte', 'classe']]

## Osservazioni da HPA, MitoCarta e fonti esterne

In [7]:
with zipfile.ZipFile(CACHE / 'hpa_subcellular.zip') as z:
    hpa = pd.read_csv(z.open(z.namelist()[0]), sep='\t')
HPA_MAP = {'Nucleoplasm': 'Nucleus', 'Nuclear speckles': 'Nucleus', 'Nuclear membrane': 'Nucleus', 'Nucleoli': 'Nucleus',
           'Nucleoli fibrillar center': 'Nucleus', 'Nucleoli rim': 'Nucleus', 'Nuclear bodies': 'Nucleus',
           'Kinetochore': 'Nucleus', 'Mitotic chromosome': 'Nucleus', 'Cytosol': 'Cytoplasm', 'Centrosome': 'Cytoplasm',
           'Centriolar satellite': 'Cytoplasm', 'Microtubules': 'Cytoplasm', 'Intermediate filaments': 'Cytoplasm',
           'Actin filaments': 'Cytoplasm', 'Aggresome': 'Cytoplasm', 'Cytoplasmic bodies': 'Cytoplasm',
           'Mitotic spindle': 'Cytoplasm', 'Plasma membrane': 'Cell membrane', 'Cell Junctions': 'Cell membrane',
           'Focal adhesion sites': 'Cell membrane', 'Endoplasmic reticulum': 'Endoplasmic reticulum',
           'Golgi apparatus': 'Golgi apparatus', 'Mitochondria': 'Mitochondrion', 'Lysosomes': 'Lysosome/Vacuole',
           'Peroxisomes': 'Peroxisome'}
hpa_long = hpa.melt(id_vars='Gene name', value_vars=['Enhanced', 'Supported', 'Approved'], var_name='rel', value_name='loc').dropna()
hpa_long = hpa_long.assign(loc=hpa_long['loc'].str.split(';')).explode('loc')
hpa_long['classe'] = hpa_long['loc'].str.strip().map(HPA_MAP)
hpa_long['fonte'] = np.where(hpa_long['rel'] == 'Approved', 'hpa_approved', 'hpa_alta')
hpa_long['uniprot_id'] = hpa_long['Gene name'].str.upper().map(sym2id)
# una proteina HPA "Approved" fa parte della fonte anche se la localizzazione è un organello non mappato (es. Vesicles):
# per questo tengo le righe con classe vuota, servono a dire che la proteina è stata misurata
obs_hpa = hpa_long.dropna(subset=['uniprot_id'])[['uniprot_id', 'fonte', 'classe']]

_mito_xlsx = CACHE / 'mitocarta3.xlsx'
_mito_html = CACHE / 'mitocarta3.html'
if _mito_xlsx.exists():
    _mito = pd.read_excel(_mito_xlsx, sheet_name='A Human MitoCarta3.0', engine='xlrd')
    mito_ids = set(_mito['Symbol'].astype(str).str.upper().map(sym2id).dropna())
elif _mito_html.exists():
    _mito = pd.read_html(io.StringIO(_mito_html.read_text(errors='ignore')))[0]
    mito_ids = set(_mito['Symbol'].astype(str).str.upper().map(sym2id).dropna())
else:
    import warnings; warnings.warn('mitocarta3.xlsx non trovato: uso fallback UniProt KW-0496 (Mitochondrion)')
    mito_ids = set(up.loc[up['keywords'].fillna('').str.contains('KW-0496', na=False), 'uniprot_id'])

# OpenCell (tagging endogeno in HEK293T): gradi 3 e 2 = localizzazione netta; il grado 1 (debole) non conta
# né come presente né come assente
OC_MAP = {'cytoplasmic': 'Cytoplasm', 'cytoskeleton': 'Cytoplasm', 'centrosome': 'Cytoplasm',
          'nucleoplasm': 'Nucleus', 'nucleolus_gc': 'Nucleus', 'nucleolus_fc_dfc': 'Nucleus', 'nuclear_punctae': 'Nucleus',
          'nuclear_membrane': 'Nucleus', 'chromatin': 'Nucleus', 'membrane': 'Cell membrane', 'cell_contact': 'Cell membrane',
          'focal_adhesions': 'Cell membrane', 'er': 'Endoplasmic reticulum', 'golgi': 'Golgi apparatus',
          'mitochondria': 'Mitochondrion'}                     # 'vesicles' e 'big_aggregates' non sono classi DeepLoc
oc = pd.read_csv(CACHE / 'opencell_localization.csv')
oc = oc.melt(id_vars='target_name', value_vars=['annotations_grade_3', 'annotations_grade_2', 'annotations_grade_1'],
             var_name='grade', value_name='loc').dropna()
oc = oc.assign(loc=oc['loc'].str.split(r'[;,]')).explode('loc')
oc['classe'] = oc['loc'].str.strip().map(OC_MAP)
oc['uniprot_id'] = oc['target_name'].str.upper().map(sym2id)
oc['fonte'] = 'opencell'
obs_oc = oc[oc['grade'] != 'annotations_grade_1'].dropna(subset=['uniprot_id'])[['uniprot_id', 'fonte', 'classe']]
oc_weak = oc[oc['grade'] == 'annotations_grade_1'].dropna(subset=['uniprot_id', 'classe'])[['uniprot_id', 'fonte', 'classe']]

# SynGO: legge ontologies.xlsx (o syngo_ontologies*.xlsx) dal bulk download di syngoportal.org.
# Usa ontologies.xlsx che contiene il rollup gerarchico: hgnc_symbol = tutti i geni del termine + figli.
# Solo termini CC: presynap*/synaptic vesicle/active zone -> Presinapsi; postsynap* -> Postsinapsi.
_sg_candidates = sorted(EXT.glob('ontologies.xlsx')) + sorted(EXT.glob('syngo_ontologies*.xlsx'))
if _sg_candidates:
    sg = pd.read_excel(_sg_candidates[0])
    sg = sg[sg['domain'] == 'CC'].copy()
    sg['classe'] = np.select(
        [sg['name'].str.contains(r'(?i)presynap|synaptic vesicle|active zone'),
         sg['name'].str.contains(r'(?i)postsynap')],
        ['Presinapsi', 'Postsinapsi'], default='nessuna')
    sg = sg[sg['classe'] != 'nessuna']
    sg = sg.assign(gene=sg['hgnc_symbol'].astype(str).str.split(r'\s*,\s*')).explode('gene')
    sg['uniprot_id'] = sg['gene'].str.strip().str.upper().map(sym2id)
    obs_syngo = sg.dropna(subset=['uniprot_id']).assign(fonte='syngo')[['uniprot_id', 'fonte', 'classe']]
else:
    obs_syngo = pd.DataFrame(columns=['uniprot_id', 'fonte', 'classe'])
print(f'OpenCell: {obs_oc["uniprot_id"].nunique():,} proteine | SynGO: {obs_syngo["uniprot_id"].nunique():,} proteine')

# fonti esterne in formato lungo (gene, classe); i simboli di topo maiuscoli coincidono quasi sempre con quelli umani
ext_files = sorted((EXT / 'fonti').glob('*.csv'))
ext = pd.concat([pd.read_csv(f).assign(fonte=f.stem) for f in ext_files] +
                [pd.DataFrame(columns=['gene', 'classe', 'fonte'])], ignore_index=True)
ext['uniprot_id'] = ext['gene'].astype(str).str.upper().map(sym2id)
obs_ext = ext.dropna(subset=['uniprot_id'])[['uniprot_id', 'fonte', 'classe']]
print('Fonti esterne trovate:', [f.stem for f in ext_files])

OpenCell: 1,309 proteine | SynGO: 1,462 proteine
Fonti esterne trovate: []


## Regola delle assenze → tabella lunga di osservazioni
Per ogni fonte: proteine riportate × classi che la fonte sa misurare. Presente = 1, altrimenti 0.
MitoCarta è un inventario dell'intero genoma: per lei ogni proteina umana assente è un'osservazione 0.

In [8]:
pos = pd.concat([obs_uniprot, obs_hpa, obs_oc, obs_syngo, obs_ext], ignore_index=True)
covered = pos[['uniprot_id', 'fonte']].drop_duplicates()
measured = pos.dropna(subset=['classe'])[['fonte', 'classe']].drop_duplicates()
grid = covered.merge(measured, on='fonte')
hits = pos.dropna(subset=['classe']).drop_duplicates().assign(obs=1.0)
obs = grid.merge(hits, on=['uniprot_id', 'fonte', 'classe'], how='left').fillna({'obs': 0.0})
# le localizzazioni OpenCell di grado 1 restano "non osservate" invece di diventare assenze
obs = obs.merge(oc_weak.drop_duplicates().assign(debole=1), on=['uniprot_id', 'fonte', 'classe'], how='left')
obs = obs[obs['debole'].isna() | (obs['obs'] == 1)].drop(columns='debole')
obs_mito = pd.DataFrame({'uniprot_id': up['uniprot_id'], 'fonte': 'mitocarta', 'classe': 'Mitochondrion',
                         'obs': up['uniprot_id'].isin(mito_ids).astype(float)})
# DeepLoc è la fonte "oro": tutte e 9 le classi osservate per ogni proteina
obs_dl = dl.melt(id_vars='uniprot_id', value_vars=CLASSES, var_name='classe', value_name='obs').assign(fonte='deeploc')
obs = pd.concat([obs_dl, obs, obs_mito], ignore_index=True)[['uniprot_id', 'fonte', 'classe', 'obs']]
print(obs.groupby('fonte').agg(proteine=('uniprot_id', 'nunique'), osservazioni=('obs', 'size'), positivi=('obs', 'sum')))

                proteine  osservazioni  positivi
fonte                                           
deeploc            28303        254727   34927.0
hpa_alta            6404         51232    7135.0
hpa_approved        7847         62776    9724.0
mitocarta          20430         20430    1132.0
opencell            1309          7624    1661.0
syngo               1462          2924    1928.0
uniprot_debole      4815         62595    6609.0
uniprot_forte      10353        134589   14203.0


## Target ausiliari (meccanismi) e contesto genico (le feature utili di FFFFF)
I target ausiliari sono proprietà della proteina che dipendono dalla sequenza e spiegano *perché* va in un
compartimento: peptide segnale (via secretoria), transit peptide (mitocondrio), eliche TM (membrane), ancore
lipidiche/GPI (membrana), NLS (nucleo), glicosilazione (passaggio ER-Golgi), tendenza ad aggregare. Il modello
li deve **predire** dalla sequenza: non sono mai input. Sul mutante, la loro variazione indica il meccanismo rotto.
Il contesto genico (multi-localizzazione, densità di varianti patogeniche, espressione cerebrale) entra solo
nel modello delle varianti.

In [9]:
aux = pd.DataFrame({'uniprot_id': up['uniprot_id'],
                    'aux_peptide_segnale': up['sig'].notna().astype(float),
                    'aux_transit_peptide': up['transit'].notna().astype(float),
                    'aux_tm_1piu': up['tm'].fillna('').str.count('TRANSMEM').ge(1).astype(float),
                    'aux_tm_2piu': up['tm'].fillna('').str.count('TRANSMEM').ge(2).astype(float),
                    'aux_ancora_lipidica': (up['lipid'].notna() | up['keywords'].fillna('').str.contains('GPI-anchor|Lipoprotein')).astype(float),
                    'aux_glicoproteina': up['keywords'].fillna('').str.contains('Glycoprotein').astype(float)})

# Feature ereditate da FFFFF, ricavate dai testi UniProt di tutte le proteine umane (non solo quelle "cerebrali")
fkw = (up['function'].fillna('') + ' ' + up['subunit'].fillna('') + ' ' + up['keywords'].fillna('')).str.lower()
ptm = (up['ptm'].fillna('') + ' ' + up['keywords'].fillna('')).str.lower()
dis = up['disease'].fillna('').str.lower()
syn_genes = set(pd.read_csv(CACHE / 'synaptopathydb_genes.csv')['HumanName'].astype(str).str.upper())
aux['aux_nls'] = (up['region'].fillna('') + up['motif'].fillna('') + up['keywords'].fillna('')).str.contains('(?i)nuclear localization signal')
aux['aux_multimero'] = fkw.str.contains(r'\bhomodimer|\bheterodimer|\bdimer|\btrimer|\btetramer|\boligomer|\bmultimer')
aux['aux_fosforilazione'] = ptm.str.contains(r'phosphorylat|phosphoprotein')
aux['aux_ubiquitinazione'] = ptm.str.contains(r'ubiquitinat|ubiquitylat|ubl conjugation')
aux['aux_palmitoilazione'] = ptm.str.contains(r'palmitoylat|palmitate')
aux['aux_processamento'] = ptm.str.contains(r'proteolytic|cleav|propeptide|zymogen')
aux['aux_aggregati'] = (dis + ' ' + ptm + ' ' + fkw).str.contains(r'amyloid|lewy bod|neurofibrillary tangle|inclusion bod|protein inclusion|aggregat')
aux['aux_patologia_cerebrale'] = dis.str.contains(r'alzheimer|parkinson|epilep|encephalopath|dementia|neurodegenerat|huntington|'
                                                  r'ataxia|intellectual disability|amyotrophic|leukodystroph|microcephal')
aux['aux_sinaptopatia'] = up['gene'].isin(syn_genes)
aux[aux.columns[1:]] = aux[aux.columns[1:]].astype(float)

# contesto genico (solo per il modello delle varianti): quanto la proteina è multi-localizzata, se fa la spola
# nucleo-citoplasma, se ha strutture risolte, che ruolo ha nel trafficking (stessa logica di FFFFF)
hpa_cls = hpa_long.dropna(subset=['classe', 'uniprot_id']).groupby('uniprot_id')['classe']
up_cls = obs_uniprot[obs_uniprot['classe'].isin(CLASSES)].groupby('uniprot_id')['classe']
ROLES = [('sorting receptor', r'sorting receptor|clathrin|endocytosis|vesicle trafficking'),
         ('channel', r'\bchannel\b|\bpore\b|\btransporter\b'),
         ('enzyme', r'\bkinase\b|\bphosphatase\b|\bprotease\b|\bligase\b|\bhydrolase\b|\bsynthase\b|\breductase\b'),
         ('receptor', r'\breceptor\b'), ('structural', r'\bstructural\b|\bscaffold\b|\bcytoskeletal\b|\bfilament\b')]
ctx_genic = pd.DataFrame({'uniprot_id': up['uniprot_id'],
                          'ctx_n_localizzazioni_hpa': up['uniprot_id'].map(hpa_cls.nunique()),
                          'ctx_n_localizzazioni_uniprot': up['uniprot_id'].map(up_cls.nunique()),
                          'ctx_shuttle_hpa': up['uniprot_id'].map(hpa_cls.agg(set).map({'Nucleus', 'Cytoplasm'}.issubset)).astype(float),
                          'ctx_shuttle_uniprot': up['uniprot_id'].map(up_cls.agg(set).map({'Nucleus', 'Cytoplasm'}.issubset)).astype(float),
                          'ctx_struttura_pdb': up['pdb'].notna().astype(float),
                          'ctx_ruolo_trafficking': np.select([fkw.str.contains(p) for _, p in ROLES], np.arange(1, 6), 6)})

with zipfile.ZipFile(CACHE / 'hpa_rna_tissue.zip') as z:
    rna = pd.read_csv(z.open(z.namelist()[0]), sep='\t', usecols=['Gene name', 'Tissue', 'nTPM'])
BRAIN_TISSUES = ['cerebral cortex', 'cerebellum', 'hippocampal formation', 'hippocampus', 'basal ganglia', 'hypothalamus',
                 'amygdala', 'midbrain', 'pons and medulla', 'pons', 'medulla oblongata', 'spinal cord', 'white matter',
                 'thalamus', 'choroid plexus']
brain_ntpm = rna[rna['Tissue'].str.lower().isin(BRAIN_TISSUES)].groupby(rna['Gene name'].str.upper())['nTPM'].max()
# espressione per tipo cellulare: i compartimenti neuronali hanno senso solo per proteine espresse nei neuroni
with zipfile.ZipFile(CACHE / 'hpa_rna_single_cell.zip') as z:
    sc = pd.read_csv(z.open(z.namelist()[0]), sep='\t', usecols=['Gene name', 'Cell type', 'nCPM'])
sc_gene = sc['Gene name'].str.upper()
neuron_ncpm = sc[sc['Cell type'].str.contains('neuron', case=False)].groupby(sc_gene)['nCPM'].max()
glia_ncpm = sc[sc['Cell type'].str.contains('astrocyte|oligodendro|microglia', case=False)].groupby(sc_gene)['nCPM'].max()

ctx = up[['uniprot_id', 'gene', 'geneid', 'protein_name']].copy()
ctx['brain_ntpm'] = ctx['gene'].map(brain_ntpm)
ctx['neuron_ncpm'] = ctx['gene'].map(neuron_ncpm)
ctx['glia_ncpm'] = ctx['gene'].map(glia_ncpm)
ctx['is_brain'] = (ctx['uniprot_id'].isin(brain_ids) | (ctx['brain_ntpm'] >= BRAIN_NTPM)).astype(int)
ctx['is_neuronal'] = (ctx['neuron_ncpm'] >= NEURON_NCPM).astype(int)
ctx = ctx.merge(aux, on='uniprot_id', how='left').merge(ctx_genic, on='uniprot_id', how='left')

## Merge finale delle proteine
DeepLoc e test HPA restano come sono. Le proteine umane non presenti in DeepLoc entrano solo se almeno una fonte
le osserva positive per un organello o un compartimento neuronale. Duplicati risolti per sequenza identica.

In [10]:
pos_ids = set(obs.loc[obs['obs'] == 1, 'uniprot_id'])
extra = up[up['uniprot_id'].isin(pos_ids) & ~up['sequence'].isin(set(dl['sequence']) | set(hpa_test['sequence']))]
extra = extra[['uniprot_id', 'sequence']].assign(source='human_extra', partition=-1)
proteins = pd.concat([dl[['uniprot_id', 'sequence', 'source', 'partition']],
                      hpa_test[['uniprot_id', 'id_test', 'sequence', 'source', 'partition'] + CLASSES[:2] + CLASSES[3:]],
                      extra], ignore_index=True)
proteins['sequence'] = proteins['sequence'].str.upper().str.replace(r'[^A-Z]', '', regex=True)
proteins = proteins[proteins['sequence'].str.len() > 40].drop_duplicates(['source', 'sequence']).reset_index(drop=True)
proteins = proteins.merge(ctx, on='uniprot_id', how='left')
proteins[['is_brain', 'is_neuronal']] = proteins[['is_brain', 'is_neuronal']].fillna(0).astype(int)
proteins['pid'] = np.arange(len(proteins))

# le osservazioni puntano alle righe di training/estese; il test HPA tiene le sue etichette nelle colonne CLASSES
train_pid = proteins[proteins['source'] != 'hpa_test'].drop_duplicates('uniprot_id').set_index('uniprot_id')['pid']
obs['pid'] = obs['uniprot_id'].map(train_pid)
obs = obs.dropna(subset=['pid']).astype({'pid': int}).drop_duplicates(['pid', 'fonte', 'classe'])

print(pd.crosstab(proteins['source'], proteins['is_brain']))
print(f"Proteine neuronali: {proteins['is_neuronal'].sum():,}")
print(obs.pivot_table(index='fonte', columns='classe', values='obs', aggfunc='sum').fillna(0).astype(int).T)
proteins.to_parquet(DATA / 'proteine.parquet', index=False)
obs.to_parquet(DATA / 'osservazioni.parquet', index=False)

is_brain         0     1
source                  
deeploc      23214  5086
hpa_test      1479   221
human_extra   4087  5950
Proteine neuronali: 11,370
fonte                  deeploc  hpa_alta  hpa_approved  mitocarta  opencell  \
classe                                                                        
Assone                       0         0             0          0         0   
Cell membrane             4187       873          1254          0       168   
Cytoplasm                 9869      2019          3195          0       607   
Dendrite                     0         0             0          0         0   
Endoplasmic reticulum     2180       231           275          0       115   
Extracellular             3299         0             0          0         0   
Golgi apparatus           1279       265           812          0        89   
Lysosome/Vacuole          1496        15             3          0         0   
Mitochondrion             2590       493           486    

## Varianti: ClinVar (missense, proteine cerebrali) e varianti con localizzazione misurata
P/LP = patogeniche, B/LB = benigne (controllo), VUS. Le "conflicting" sono escluse. La variante si tiene solo se il
residuo WT coincide con la sequenza UniProt (la posizione p. è riferita al trascritto RefSeq).

In [11]:
brain = proteins[(proteins['is_brain'] == 1) & proteins['geneid'].notna() & (proteins['source'] != 'hpa_test')]
gene2prot = brain.assign(GeneID=brain['geneid'].str.strip(';').str.split(';')).explode('GeneID')
gene2prot['GeneID'] = pd.to_numeric(gene2prot['GeneID'], errors='coerce')
gene2prot = gene2prot.dropna(subset=['GeneID']).drop_duplicates('GeneID').set_index('GeneID')[['pid', 'gene', 'sequence']]

CLIN = {'Pathogenic': 'P/LP', 'Likely pathogenic': 'P/LP', 'Pathogenic/Likely pathogenic': 'P/LP',
        'Benign': 'B/LB', 'Likely benign': 'B/LB', 'Benign/Likely benign': 'B/LB', 'Uncertain significance': 'VUS'}
AA3 = {'Ala': 'A', 'Arg': 'R', 'Asn': 'N', 'Asp': 'D', 'Cys': 'C', 'Gln': 'Q', 'Glu': 'E', 'Gly': 'G', 'His': 'H',
       'Ile': 'I', 'Leu': 'L', 'Lys': 'K', 'Met': 'M', 'Phe': 'F', 'Pro': 'P', 'Ser': 'S', 'Thr': 'T', 'Trp': 'W',
       'Tyr': 'Y', 'Val': 'V'}
chunks = []
cols_cv = ['Type', 'Name', 'GeneID', 'ClinicalSignificance', 'Assembly', 'PhenotypeList', 'VariationID']
with gzip.open(CACHE / 'clinvar_variant_summary.txt.gz', 'rt') as fh:
    for ch in pd.read_csv(fh, sep='\t', usecols=cols_cv, chunksize=250_000, low_memory=False):
        ch = ch[(ch['Assembly'] == 'GRCh38') & (ch['Type'] == 'single nucleotide variant') & ch['GeneID'].isin(gene2prot.index)]
        chunks.append(ch.assign(clin=ch['ClinicalSignificance'].map(CLIN)).dropna(subset=['clin']))
cv = pd.concat(chunks, ignore_index=True)
cv[['wt', 'pos', 'mt']] = cv['Name'].str.extract(r'\(p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})\)')
cv['wt'], cv['mt'] = cv['wt'].map(AA3), cv['mt'].map(AA3)
cv = cv.dropna(subset=['wt', 'mt']).join(gene2prot, on='GeneID')
cv = cv[cv['wt'] != cv['mt']].astype({'pos': int})
cv = cv[cv['wt'] == [s[p - 1:p] for s, p in zip(cv['sequence'], cv['pos'])]]
cv['brain_phenotype'] = cv['PhenotypeList'].str.contains(
    r'(?i)brain|cerebr|neuro|epilep|seizure|ataxi|dementia|alzheimer|parkinson|intellectual|autism|encephal|'
    r'leukodystroph|microceph|spastic|schizophren', na=False).astype(int)
cv['variante'] = cv['wt'] + cv['pos'].astype(str) + cv['mt']
cv = cv.drop_duplicates(['pid', 'pos', 'mt', 'clin'])[['pid', 'gene', 'VariationID', 'variante', 'wt', 'pos', 'mt', 'clin', 'brain_phenotype']]

# contesto genico ricalcolato (in FFFFF il download ClinVar era fallito): quota di varianti patogeniche per gene
cv_gene = cv.groupby('pid')['clin'].value_counts().unstack(fill_value=0).reindex(columns=['P/LP', 'B/LB', 'VUS'], fill_value=0)
cv['ctx_frazione_patogeniche_gene'] = cv['pid'].map((cv_gene['P/LP'] + 1) / (cv_gene.sum(axis=1) + 2))
cv.to_parquet(DATA / 'clinvar_missense.parquet', index=False)
print(cv['clin'].value_counts().to_string())

var_files = sorted((EXT / 'varianti').glob('*.csv'))
lab_var = pd.concat([pd.read_csv(f) for f in var_files] + [pd.DataFrame(columns=['gene', 'variante', 'mislocalizzata'])],
                    ignore_index=True)[['gene', 'variante', 'mislocalizzata']]
lab_var['gene'] = lab_var['gene'].astype(str).str.upper()
lab_var.to_parquet(DATA / 'varianti_sperimentali.parquet', index=False)
print(f'Varianti con localizzazione misurata: {len(lab_var):,}')

clin
VUS     1248170
B/LB      77890
P/LP      46814
Varianti con localizzazione misurata: 0
